# DEMO PIPELINE IN CIFAR10

In [1]:
#import necessary libraries
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision.models import resnet18
from torch.utils.data import DataLoader

# 1. LOAD DATA

In [1]:
from deepcore.datasets import CIFAR10

#load the dataset
channel, im_size, num_classes, class_names, mean, std, dst_train, dst_test = CIFAR10('./data')


Files already downloaded and verified
Files already downloaded and verified


# 3. DEEPCORE

In [2]:
# IMPORT METHOD
from deepcore.methods import Herding

In [ ]:
#dùng cái này nếu muốn sử dụng các bộ data khác mà thư viện không cung cấp(viết thay vào dst_train ở hàm select)

from torch.utils.data import Dataset
from torch.nn.functional import  one_hot


class CustomDataset(Dataset):
    def __init__(self, x:torch.tensor, y:torch.tensor, classes:list):
        """
        Args:
            x (tensor): Dữ liệu đầu vào có shape (N, H, W, C).
            y (tensor): Nhãn dạng one-hot hoặc chỉ số lớp (N,).
            classes (list): Danh sách các nhãn lớp (nếu có).
        """
        self.x = x.permute(0,3,1,2).float()  # Đổi shape thành (N, C, H, W)
        self.y = one_hot(y, num_classes=len(classes))  # Chuyển one-hot thành chỉ số lớp
        self.classes = classes 

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]


In [ ]:
#định nghĩa class args bao gồm các tham số để chạy method
class Args:
    def __init__(self, model='ResNet18', channel=3, num_classes=43, im_size=(32, 32), selection_batch=128,
                 print_freq=100, workers=4, device='cuda', selection_method="LeastConfidence",
                 selection_optimizer="Adam", selection_lr=1e-3, selection_momentum=0.0, selection_weight_decay=1e-5,selection_nesterov=False, **kwargs):
        self.model = model
        self.channel = channel
        self.num_classes = num_classes
        self.im_size = im_size
        self.selection_batch = selection_batch
        self.print_freq = print_freq
        self.device = device
        self.selection_method = selection_method
        self.selection_optimizer = selection_optimizer
        self.selection_lr = selection_lr
        self.selection_nesterov = selection_nesterov
        self.selection_momentum = selection_momentum
        self.selection_weight_decay = selection_weight_decay
        if device == 'cuda': 
            self.gpu = kwargs.get('gpu', None)
            self.workers = workers
        if self.gpu is None:
            self.workers = 0
            self.device = 'cpu'

In [ ]:
fraction = 0.3
args = Args(model='ResNet18', channel=3, num_classes=10, im_size=(32,32), selection_batch=256,
            print_freq=100, workers=4, device='cuda') 
herd = Herding(dst_train=dst_train, args=args, fraction=fraction, random_seed=42, epochs=20)

model1, result = herd.select()
selected_indices = result["indices"]

SyntaxError: expected argument value expression (1383237207.py, line 2)

In [ ]:
#sử dụng cpu
fraction = 0.3
args = Args(model='ResNet18', channel=3, num_classes=10, im_size=(32, 32), selection_batch=128,
            print_freq=100, device='cpu') # Example: using GPU 0
herd = Herding(dst_train=dst_train, args=args, fraction=fraction, random_seed=1, epochs=20)

model1, result = herd.select()
selected_indices = result["indices"]

Using CPU.

=> Training Epoch #0
| Epoch [  0/  1] Iter[  1/391]		Loss: 2.4429
| Epoch [  0/  1] Iter[101/391]		Loss: 1.3756
| Epoch [  0/  1] Iter[201/391]		Loss: 1.3041
| Epoch [  0/  1] Iter[301/391]		Loss: 1.1624
not balance


KeyboardInterrupt: 

# 4. TEST

In [ ]:
# Create a DataLoader for the selected indices and test set
from torch.utils.data import  Subset


subset_dataset = Subset(dst_train, selected_indices)


subset_dataloader = DataLoader(
    subset_dataset,
    batch_size=256,  # Use the batch size from your arguments
    shuffle=True,
    num_workers=args.workers  # Use the number of workers from your arguments
)

valloader = DataLoader(dst_test, batch_size=256, shuffle=True)

In [ ]:

# Kiểm tra GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
model = resnet18(num_classes=10).to(device)

# Loss function và optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.005)
num_epochs = 40


In [ ]:
#train
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    for images, labels in subset_dataloader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {running_loss/len(subset_dataloader):.4f}")

print("Training complete!")


In [ ]:
#validation
model.eval()
correct, total = 0, 0

with torch.no_grad():
    for images, labels in valloader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        # Compare predicted class indices with actual class indices (labels) directly
        correct += (predicted == labels).sum().item()  

print(f"Accuracy: {100 * correct / total:.2f}%")